# DeepStream Pow Wow
These are my notebooks while going through an NVIDIA DeepStream SDK self learning. 

Opportunities for Video AI or Intelligent Video Analytics (IVA) 
<p><img src='images/use_cases.png' width='720px' /></p>

* **Access control** at airports or other checkpoints
* **Managing operations** for logistic and manufacturing such as warehouse balancing at product distribution centers
* **Traffic flow and parking management** for _smart cities_
* **Retail analytics** to improve customer experience
* **Optical inspection** at factory assembly line

<a name='s1.1'></a>
### Video AI Tasks ###
In a typical Video Analytics pipeline, there are one or more machine learning models to generate insights from the video inputs. These are generally deep learning neural network (DNN) models that have been trained for a specific visual task. There are numerous approaches for drawing insight from videos: 
* **Classification** is used for identifying the object contained in an image. It is the task of labeling the given frame with one of the classes that the model has been trained with
* **Localization** uses regression to return the coordinates of the potential object within the frame
* **Object detection**, which includes _image localization_, can specify the location of multiple objects in a frame
* **Segmentation** provides pixel level accuracy by creating a fine-grained segmentation mask around the detected object. Applications for segmentation include: an AI powered green screen to blur or change the background of the frame, autonomous driving where you want to segment the road and background, or for manufacturing to identify microscopic level defects

<p><img src='images/image_processing_problems.png' width='720px'></p>

<a name='s2'></a>
## A typical Video AI Pipeline
Generally, a video AI pipeline takes one or more input video streams, performs decoding and muxing (or aggregating), preprocesses the batch, and puts the data through AI inference. Afterwards, the AI-based insight, combined with the original video, can be 1) encoded for storage, 2) used to create a composite for display, or 3) passed downstream for further analytics.
<p><img src='images/iva_framework.png' width=720></p>

# Lets talk DeepStream SDK #
The DeepStream SDK is a streaming analytics toolkit that can be used to build video AI applications or pipeline.

<a name='s1.1'></a>
### Video Formats ###
The input video file is an encoded video file with a **.h264** extension, which is perhaps not the **.mp4** extension we would expect for a video file. The .mp4 file extension is a representation of the container, which has all the files needed to play back a video. These files include the visual images, the audio tracks, and the metadata (i.e., bitrate, resolution, subtitles, timestamp, etc.). The metadata also contains information about the **codec** used for the audio and video streams. The codec, which is a mashup of the words *co*de and *dec*ode, is a method used to compress (encode) a video into a smaller size for faster transmission. The encoded file can be decompressed (decoded) using the same codec for playback and processing. The most common video codecs include **[H.264](https://en.wikipedia.org/wiki/Advanced_Video_Coding)**, **[H.265](https://en.wikipedia.org/wiki/High_Efficiency_Video_Coding)**, and **[MPEG4](https://en.wikipedia.org/wiki/MPEG-4)**. Separate from MPEG4, **[MP4](https://en.wikipedia.org/wiki/MPEG-4_Part_14)** is a container that can be used for playback in the JupyterLab. These properties describe the video format and new ones are continuously being developed to provide improvements in quality, file size, and video playback. We need to build the application based on the video format(s) of the input and desire output. 

The [FFmpeg](https://ffmpeg.org/) tool is a very fast video and audio converter with the general syntax: <br> `ffmpeg [global_options] {[input_file_options] -i input_url} ... {[output_file_options] output_url} ...`. <br> When using the `ffmpeg` command, the `-i` option lets us read an input URL, the `-loglevel quiet` option suppresses the logs, and the `-y` flag overwrites any existing output file with the same name. 

In [ ]:
from IPython.display import Video

# Convert the H.264 encoded video file to MP4 container file - this will generate the sample_30.mp4 file
!ffmpeg -i /dli/task/data/sample_30.h264 /dli/task/sample_30.mp4 \
        -y \
        -loglevel quiet

# View the input video
Video('sample_30.mp4', width=720)

<a name='s2'></a>
## GStreamer Foundations ##
DeepStream utilizes an optimized graph architecture built using the open-source [GStreamer multimedia framework](https://gstreamer.freedesktop.org/). GStreamer is used for creating streaming media applications, ranging from a simple media player to complex video editing applications. GStreamer plugins can be mixed and matched into arbitrary pipelines to create custom applications. 

### Key concepts in GStreamer 
* **Elements** - Elements are at the core of GStreamer. Elements provide some sort of functionality when linked with other elements. For example, a source element provides data to a stream, a filter element processes a stream of data, and a sink element consumes data. Data flow downstream from source elements to sink elements, passing through filter elements. GStreamer offers a large collection of elements by default but also allows for writing new elements. 
    * A [sink](https://en.wikipedia.org/wiki/Sink_(computing)), in computing, is designed to receive data. 
* **Bins** - Bins are container elements that allow you to combine linked elements into a logical group. Bins can be handled in the same way as any other element. They are programmed to manage elements contained within, including state changes as well as bus messages, to ensure that data flow smoothly. This is useful when constructing complex pipelines that require many elements. 
* **Pipeline** - A pipeline is the top-level bin that also manages the synchronization and bus messages of the contained elements. 
* **Plugins** - Elements need to be encapsulated in a plugin to enable GStreamer to use it. A plugin is essentially a loadable block of code, usually recognized as a shared object file or a dynamically linked library. A plugin may contain the implementation of several elements, or just one. GStreamer provides building blocks in the form of plugins that can be used to construct an efficient video analytics pipeline. The DeepStream SDK features hardware-accelerated plugins that bring deep neural networks and other complex processing tasks into the stream processing pipeline. 
* **Bus** - The bus is the object responsible for delivering to the application **messages** generated by the elements. Every pipeline contains a bus by default, so the only thing applications should do is set a message handler on a bus, which is like a signal handler to an object. When the main loop is running, the bus will periodically be checked for new messages, and the message handler will be called when any new message is available. 
    * Messages signal the application of pipeline events. Some of the message types include `GST_MESSAGE_EOS` (end-of-stream), `GST_MESSAGE_ERROR`, and `GST_MESSAGE_WARNING`. 
* **Pads** - Pads are used to negotiate links and dataflow between elements in GStreamer. A pad is the “port” on an element where links can be made with other elements for data to flow through. When data flow from element to element in a pipeline, in reality it flows from the source pad of one element to the sink pad of another. Links are only allowed between two pads when the data types, or **capabilities**, are compatible. 
* **Buffers** and **Events** - All streams of data in GStreamer are chopped up into chunks and passed from a source pad of one element to a sink pad of another element as one of the two types of `GstMiniObject`: **events** (control) and **buffers** (content). A buffer is the basic unit of data transfer in GStreamer. Normally, it contains a chunk of video data that flow from one element to another. The DeepStream SDK attaches the DeepStream metadata object, `NvDsBatchMeta`, to the buffer. An event, on the other hand, contains information on the state of the stream flowing between two linked pads. Events can be used to indicate the end of a media stream. 
* **Queries** - Queries are used to get information about the stream. 

>For the most part, all data in GStreamer flow one way through a link between elements. When data flow from one DeepStream element to another, the buffers are not recreated. Instead, buffer pointers are passed to avoid unnecessary copies and achieve high-speed performance. 

<p><img src='images/gstreamer.png' width='720px'></p>

For more information, please refer to [GStreamer Basics](https://gstreamer.freedesktop.org/documentation/application-development/basics/index.html). 

<a name='s3'></a>
## Anatomy of a DeepStream Pipeline ##
GStreamer and by extension DeepStream applications have a **plugin-based architecture**. One single **plugin** may contain the implementation of several elements, or just one. When building a pipeline, we can select from a catalogue of available [GStreamer](https://gstreamer.freedesktop.org/documentation/plugins_doc.html) or [DeepStream](https://docs.nvidia.com/metropolis/deepstream/dev-guide/text/DS_plugin_Intro.html#) plugins, or create new ones. An application can be thought of as a pipeline consisting of individual components (plugins), each representing a functional block like video decoding/encoding, scaling, inferencing, and more. 

The graph below shows the pipeline of a typical video analytics pipeline, starting from consuming input videos to outputting insights. All the individual blocks are various plugins that are used. At the bottom are the different hardware engines that are utilized throughout the pipeline. Where applicable, plugins are accelerated using the underlying hardware to deliver maximum performance. 
<p><img src='images/deepstream_overview_graph_architecture.png' width='720px'></p>

<a name='s3.1'></a>
### Inspecting Plugins ### 
We can inspect plugins using `gst-inspect-1.0`. It's a tool that prints out information on available plugins, information about a particular plugin, or information about a particular element. When executed with no *plugin* or *element* argument, it will print a list of all plugins and elements together with a summary. When executed with a *plugin* or *element* argument, it will print information about that plugin or element.

In [ ]:
!gst-inspect-1.0

adpcmdec:  adpcmdec: ADPCM decoder
inter:  interaudiosrc: Internal audio source
inter:  interaudiosink: Internal audio sink
inter:  intersubsrc: Internal subtitle source
inter:  intersubsink: Internal subtitle sink
inter:  intervideosrc: Internal video source
inter:  intervideosink: Internal video sink
audiotestsrc:  audiotestsrc: Audio test source
faceoverlay:  faceoverlay: faceoverlay
taglib:  id3v2mux: TagLib-based ID3v2 Muxer
taglib:  apev2mux: TagLib-based APEv2 Muxer
autoconvert:  autoconvert: Select convertor based on caps
autoconvert:  autovideoconvert: Select color space convertor based on caps
speex:  speexenc: Speex audio encoder
speex:  speexdec: Speex audio decoder
fluidsynthmidi:  fluiddec: Fluidsynth
wildmidi:  wildmididec: WildMidi-based MIDI music decoder
gaudieffects:  burn: Burn
gaudieffects:  chromium: Chromium
gaudieffects:  dilate: Dilate
gaudieffects:  dodge: Dodge
gaudieffects:  exclusion: Exclusion
gaudieffects:  solarize: Solarize
gaudieffects:  gaussianblur: 

There are numerous plugins available for developers to use. You can learn more about them in the documentations for [GStreamer Plugins](https://gstreamer.freedesktop.org/documentation/plugins_doc.html) and [DeepStream Plugins](https://docs.nvidia.com/metropolis/deepstream/dev-guide/text/DS_plugin_Intro.html#). Let's now inspect a specific plugin to learn more about it. 

In [ ]:
!gst-inspect-1.0 h264parse

Factory Details:
  Rank                     primary + 1 (257)
  Long-name                H.264 parser
  Klass                    Codec/Parser/Converter/Video
  Description              Parses H.264 streams
  Author                   Mark Nauwelaerts <mark.nauwelaerts@collabora.co.uk>

Plugin Details:
  Name                     videoparsersbad
  Description              videoparsers
  Filename                 /usr/lib/x86_64-linux-gnu/gstreamer-1.0/libgstvideoparsersbad.so
  Version                  1.16.3
  License                  LGPL
  Source module            gst-plugins-bad
  Source release date      2020-10-21
  Binary package           GStreamer Bad Plugins (Ubuntu)
  Origin URL               https://launchpad.net/distros/ubuntu/+source/gst-plugins-bad1.0

GObject
 +----GInitiallyUnowned
       +----GstObject
             +----GstElement
                   +----GstBaseParse
                         +----GstH264Parse

Pad Templates:
  SRC template: 'src'
    Availability: Always


Let's inspect a DeepStream-specific plugin: `nvinfer`. 

In [12]:
!gst-inspect-1.0 nvinfer

Factory Details:
  Rank                     primary (256)
  Long-name                NvInfer plugin
  Klass                    NvInfer Plugin
  Description              Nvidia DeepStreamSDK TensorRT plugin
  Author                   NVIDIA Corporation. Deepstream for Tesla forum: https://devtalk.nvidia.com/default/board/209

Plugin Details:
  Name                     nvdsgst_infer
  Description              NVIDIA DeepStreamSDK TensorRT plugin
  Filename                 /usr/lib/x86_64-linux-gnu/gstreamer-1.0/deepstream/libnvdsgst_infer.so
  Version                  6.3.0
  License                  Proprietary
  Source module            nvinfer
  Binary package           NVIDIA DeepStreamSDK TensorRT plugin
  Origin URL               http://nvidia.com/

GObject
 +----GInitiallyUnowned
       +----GstObject
             +----GstElement
                   +----GstBaseTransform
                         +----GstNvInfer

Pad Templates:
  SRC template: 'src'
    Availability: Always
    Capa

In [13]:
!gst-inspect-1.0 nvv4l2decoder

Factory Details:
  Rank                     primary + 11 (267)
  Long-name                NVIDIA v4l2 video decoder
  Klass                    Codec/Decoder/Video
  Description              Decode video streams via V4L2 API
  Author                   Nicolas Dufresne <nicolas.dufresne@collabora.com>, Viranjan Pagar <vpagar@nvidia.com>

Plugin Details:
  Name                     nvvideo4linux2
  Description              Nvidia elements for Video 4 Linux
  Filename                 /usr/lib/x86_64-linux-gnu/gstreamer-1.0/deepstream/libgstnvvideo4linux2.so
  Version                  1.14.0
  License                  LGPL
  Source module            nvvideo4linux2
  Binary package           nvvideo4linux2
  Origin URL               http://nvidia.com/

GObject
 +----GInitiallyUnowned
       +----GstObject
             +----GstElement
                   +----GstVideoDecoder
                         +----GstNvV4l2VideoDec
                               +----nvv4l2decoder

Pad Templates:
  SINK 

The `nvinfer` plugin does inferencing on input data using NVIDIA TensorRT. It can perform AI inference on (batched) images for classification, object detection, and segmentation tasks based on the trained model we provide. There are several properties that can be set related to the inference engine, including the `model-engine-file` property. We recommend setting properties via a configuration file through the `config-file-path` property. More information about DeepStream plugins can be found in the [DeepStream Plugin Guide](https://docs.nvidia.com/metropolis/deepstream/dev-guide/index.html#plugins-development-guide). 

<a name='s4'></a>
## Accessing DeepStream MetaData ##
`GstBuffer` is the basic unit of data transfer in GStreamer. As it's passing through the pipeline, metadata received by each component is attached to the buffer. Similarly, the DeepStream SDK attaches the DeepStream metadata object, `NvDsBatchMeta` to it. DeepStream metadata contains inference results from `Gst-nvinfer` and information from other plugins in the pipeline. DeepStream uses an extensible standard structure for metadata, starting with the batch level metadata (`NvDsBatchMeta`) created inside the `Gst-nvstreammux` plugin. Subsidiary metadata structures hold frame, object, classifier, and display data. The metadata format is described in detail in the [SDK MetaData Documentation and API Guide](https://docs.nvidia.com/metropolis/deepstream/dev-guide/text/DS_plugin_metadata.html). Having some familiarity with the metadata structure will help us extract the desired information.  

<a name='s4.1'></a>
### Probe ###
<p><img src='images/probe.png' width=720></p>

We use [probes](https://gstreamer.freedesktop.org/documentation/application-development/advanced/pipeline-manipulation.html#using-probes) to access this metadata. Probing is best envisioned as having access to a pad listener. We can use them to access metadata at various points in the pipeline. Technically, a probe is a [callback function](https://en.wikipedia.org/wiki/Callback_(computer_programming)) that can be attached to a pad. While attached, the probe notifies when there is data passing on a pad. It allows us to easily interact with the data flowing through our pipeline. For more information on `GstPad` and probes, please visit GStreamer’s API Reference on [GstPad](https://gstreamer.freedesktop.org/documentation/gstreamer/gstpad.html?gi-language=c). 

>Since the video AI application will rely heavily on the metadata generated from the deep learning models, the probe callback function might be the most important piece when constructing a DeepStream pipeline. 